### 实验：config 如何从 graph。invoke() 自动传递到每个节点内部


In [21]:
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import MessagesState
from langchain_core.runnables import RunnableConfig
from typing import TypedDict, Annotated, List
import operator
import json

In [12]:
class State(MessagesState):
    trace_log : Annotated[List, operator.add]
    

In [14]:
def node_alpha(state: State, config: RunnableConfig) -> dict:
    """节点 Alpha：记录它收到的 config"""
    user_id = config.get("configurable", {}).get("user_id", "unknown")
    run_name = config.get("run_name", "unnamed")
    tags = config.get("tags", [])
    
    log_entry = {
        "node": "alpha",
        "user_id": user_id,
        "run_name": run_name,
        "tags": tags,
        "config_keys": list(config.keys()),
    }
    
    print(f"\n[Node Alpha] 收到 config:")
    print(json.dumps(log_entry, indent=2, ensure_ascii=False))
    
    return {
        "messages": [{"role": "system", "content": "alpha executed"}],
        "trace_log": [log_entry],
    }

In [15]:
def node_beta(state: State, config: RunnableConfig) -> dict:
    """节点 Beta：验证它收到相同的 configurable 但不同的 run_name"""
    user_id = config.get("configurable", {}).get("user_id", "unknown")
    run_name = config.get("run_name", "unnamed")
    
    log_entry = {
        "node": "beta",
        "user_id": user_id,  # 应该与 alpha 相同
        "run_name": run_name,  # 应该不同（LangGraph 会 patch）
        "messages_received": len(state["messages"]),
    }
    
    print(f"\n[Node Beta] 收到 config:")
    print(json.dumps(log_entry, indent=2, ensure_ascii=False))
    
    return {
        "messages": [{"role": "system", "content": "beta executed"}],
        "trace_log": [log_entry],
    }


In [16]:
builder = StateGraph(State)
builder.add_node("alpha", node_alpha)
builder.add_node("beta", node_beta)
builder.add_edge("alpha", "beta")
builder.add_edge("beta", END)
builder.set_entry_point("alpha")
graph = builder.compile()

In [22]:
# 运行实验
print("=" * 50)
print("实验：观察 Config 传播")
print("=" * 50)

config = {
    'configurable':{
        'user_id':'alice-001',
        'session_id':'sees001'
    },
    'tags': ['experiment', 'tracing_test'],
    "run_name": "config-propagation-experiment",
    "metadata": {"experiment_id": "exp-001"},
}


result = graph.invoke(
    {
        'messages':[],
        'trace_log':[]
    },
    config = config
)

print("\n" + "=" * 50)
print("最终 trace_log:")
for entry in result["trace_log"]:
    print(json.dumps(entry, indent=2, ensure_ascii=False))

实验：观察 Config 传播

[Node Alpha] 收到 config:
{
  "node": "alpha",
  "user_id": "alice-001",
  "run_name": "unnamed",
  "tags": [
    "experiment",
    "tracing_test"
  ],
  "config_keys": [
    "tags",
    "metadata",
    "configurable",
    "callbacks"
  ]
}

[Node Beta] 收到 config:
{
  "node": "beta",
  "user_id": "alice-001",
  "run_name": "unnamed",
  "messages_received": 1
}

最终 trace_log:
{
  "node": "alpha",
  "user_id": "alice-001",
  "run_name": "unnamed",
  "tags": [
    "experiment",
    "tracing_test"
  ],
  "config_keys": [
    "tags",
    "metadata",
    "configurable",
    "callbacks"
  ]
}
{
  "node": "beta",
  "user_id": "alice-001",
  "run_name": "unnamed",
  "messages_received": 1
}
